# Merging a LoRA Adapter into the Base Model (Encoder)

After fine tuning produces a LoRA adapter and the adapter passes evaluation, the final deployment step is to merge the adapter weights back into the base model. This produces a single standalone model that can be served with any standard Hugging Face inference pipeline without needing the PEFT library at runtime.

This module covers the encoder (sequence classification) variant of the merge. The base model is an `AutoModelForSequenceClassification` rather than a causal language model, which means the classification head needs special handling.

## What you will learn
- Why merging matters for production deployment
- How to reconstruct the label mapping so the classification head dimensions match the adapter
- How to load the base model on CPU (no GPU required for merging)
- How `merge_and_unload` fuses LoRA weights into the base model
- How to save the merged model for downstream serving

---
## 0 Memory Check

The encoder merge runs on CPU. Encoder models are small enough that system RAM is sufficient. The cell below prints available memory so you can confirm the workbench has headroom before starting.

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 264.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 717.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 610.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 520.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 689.5 MB/s  0:00:00
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [hf-xet]
    Found existing installation: fsspec 2026.1.0━━━━━━━━━━━━━━  3/14 [hf-xet]
    Uninstalling fsspec-2026.1.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [hf-xet]
      Successfully uninstalled fsspec-2026.1.0━━━━━━━━━━━━━━━━  3/14 [hf-xet]
  Attempting uninstall: dill╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Found existing installation: dill 0.3.9━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Uninstalling dill-0.3.9:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
      Successfully uninstalled dill-0.3.9━━━━━━━━━━━━━━━━━━━━━━━━━  5/14 [dill]


In [2]:
import shutil

total, used, free = shutil.disk_usage("/")
print(f"disk free: {free / 1024**3:.1f} GB")

import psutil
mem = psutil.virtual_memory()
print(f"RAM total: {mem.total / 1024**3:.1f} GB")
print(f"RAM available: {mem.available / 1024**3:.1f} GB")

disk free: 29.0 GB
RAM total: 15.3 GB
RAM available: 12.0 GB


---
## 1 Why Merge?

During fine tuning with LoRA, the original model weights stay frozen. The adapter only stores small low rank delta matrices that are applied on top of the base weights at inference time. This has two implications:

1. At inference time, the PEFT library must be present to load and apply the adapter. This adds a dependency and slightly increases startup latency.
2. Each forward pass computes `base_weight + adapter_delta` on the fly, adding a small overhead.

Merging bakes the delta directly into the base weights: `merged_weight = base_weight + adapter_delta`. The result is a standard Hugging Face model that loads and runs like any other checkpoint, with no PEFT dependency and no per forward pass overhead.

The merge itself is a pure arithmetic operation (matrix addition), so it does not need a GPU and runs quickly on CPU.

---
## 2 Imports

| Library | Purpose |
|---|---|
| `transformers` | Load the base encoder model (`AutoModelForSequenceClassification`) and tokenizer |
| `peft` | Load the LoRA adapter and perform the merge (`PeftModel`) |
| `torch` | Specify dtype for model loading |
| `json` | Parse the training data to reconstruct label mappings |

In [3]:
import torch
import os
import gc
import json
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

---
## 3 Configuration

Three paths are needed:
- `base_model`: the Hugging Face model ID or local path for the pre trained encoder
- `adapter_path`: where the LoRA adapter checkpoint was saved after training
- `output_path`: where the merged model will be written

The original script reads these from command line arguments. In the notebook we set them as variables so you can adjust them for your environment.

In [4]:
# update these paths for your environment
base_model_id = os.environ.get("BASE_MODEL", "clicknext/phayathaibert")
adapter_path = os.environ.get("ADAPTER_PATH", "/mnt/data/adapters/latest")
output_path = os.environ.get("OUTPUT_PATH", "/mnt/data/adapter/merged_model")
raw_training_data = os.environ.get("TRAINING_DATA", "/mnt/data/training_dataset.json")

print(f"base model:    {base_model_id}")
print(f"adapter:       {adapter_path}")
print(f"output:        {output_path}")
print(f"training data: {raw_training_data}")

base model:    clicknext/phayathaibert
adapter:       /mnt/data/adapters/latest
output:        /mnt/data/adapter/merged_model
training data: /mnt/data/training_dataset.json


---
## 4 Reconstructing the Label Mapping

This is the most important step that is easy to get wrong. An encoder model with a classification head (`AutoModelForSequenceClassification`) has a final linear layer whose output dimension equals the number of classes. During training, the model was initialized with `num_labels=N` which created this layer.

When we load the base model for merging, we must use the exact same `num_labels` so that the classification head dimensions match what the adapter expects. If the numbers do not match, the merge will fail with a shape mismatch error.

We reconstruct the mapping by scanning the same training data file that was used during fine tuning and collecting all unique intent labels. Sorting ensures the mapping is deterministic regardless of the order intents appear in the data.

In [5]:
print("extracting full label mapping to size the base model...")
all_intents = set()
with open(raw_training_data, 'r') as f:
    data = json.load(f)
    for row in data:
        all_intents.add(row.get("intent", "UNKNOWN"))

unique_intents = sorted(list(all_intents))
num_labels = len(unique_intents)

# bidirectional mappings for the classification head
id2label = {i: label for i, label in enumerate(unique_intents)}
label2id = {label: i for i, label in enumerate(unique_intents)}

print(f"found {num_labels} unique intents:")
for idx, label in id2label.items():
    print(f"  {idx}: {label}")

extracting full label mapping to size the base model...
found 16 unique intents:
  0: MOBILE_BILLING_CHECK_DUE_DATE
  1: MOBILE_BILLING_PAYMENT_STATUS_POSTPAID
  2: MOBILE_BILLING_PAYMENT_STATUS_PREPAID
  3: MOBILE_NETWORK_CHECK_NO_SIGNAL
  4: MOBILE_PACKAGE_CHECK_DATA_CURRENT
  5: MOBILE_PACKAGE_CHECK_DATA_HISTORY
  6: MOBILE_ROAMING_CHECK_DATA_CURRENT
  7: MOBILE_ROAMING_CHECK_PAYMENT_POSTPAID
  8: MOBILE_SIM_ACTIVATION_POSTPAID
  9: MOBILE_SIM_ACTIVATION_PREPAID
  10: MOBILE_SIM_REPLACEMENT_POSPAID
  11: MOBILE_SIM_REPLACEMENT_PREPAID
  12: MOBILE_USAGE_CHECK_DATA_CURRENT
  13: MOBILE_USAGE_COMPARE_DATA_PLAN
  14: MOBILE_VAS_SUBSCRIBE_DATA_HISTORY
  15: MOBILE_VAS_SUBSCRIBE_DATA_SPECIFIC_MONTH


---
## 5 Loading the Base Model on CPU

The merge is pure arithmetic (adding the LoRA delta matrices to the base weights), so it does not require a GPU. Loading on CPU with `float32` precision keeps things simple and avoids quantization artifacts.

Key parameters:
- `num_labels`, `id2label`, `label2id`: these configure the classification head to match the adapter
- `device_map="cpu"`: explicitly forces CPU placement
- `torch_dtype=torch.float32`: standard precision for CPU operations. Using float16 on CPU is slower and less numerically stable
- `trust_remote_code=True`: needed for models like Qwen that include custom code in the checkpoint

In [6]:
print(f"loading base model {base_model_id} on CPU...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    device_map="cpu",
    torch_dtype=torch.float32,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)

print(f"model loaded. classification head output size: {base_model.config.num_labels}")
print(f"model dtype: {next(base_model.parameters()).dtype}")
print(f"model device: {next(base_model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!


loading base model clicknext/phayathaibert on CPU...


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at clicknext/phayathaibert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model loaded. classification head output size: 16
model dtype: torch.float32
model device: cpu


---
## 6 Loading the LoRA Adapter

`PeftModel.from_pretrained` loads the adapter checkpoint and attaches it to the base model. At this point the model has two sets of weights:
- The original frozen base weights
- The LoRA low rank matrices (A and B) that will be merged in the next step

You can inspect the adapter's rank and target modules from the loaded config.

In [7]:
print(f"loading LoRA adapter from {adapter_path}...")
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

# show adapter details
peft_config = peft_model.peft_config["default"]
print(f"adapter rank (r): {peft_config.r}")
print(f"adapter alpha: {peft_config.lora_alpha}")
print(f"target modules: {peft_config.target_modules}")
print(f"trainable params: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}")

loading LoRA adapter from /mnt/data/adapters/latest...
adapter rank (r): 16
adapter alpha: 32
target modules: {'value', 'query'}
trainable params: 602,896


---
## 7 Merge and Unload

This is the core operation. `merge_and_unload()` does two things:

1. For each target module, it computes `W_merged = W_base + (alpha/r) * B @ A` where A and B are the low rank LoRA matrices. This folds the adapter's learned adjustments directly into the base weight matrices.
2. It removes the PEFT wrapper so the result is a plain `transformers` model with no adapter layers.

After this call, the model behaves identically to one that was fully fine tuned (from a forward pass perspective) but was trained with far fewer parameters.

In [8]:
print("fusing LoRA weights into the base model...")
merged_model = peft_model.merge_and_unload()

# free intermediate refs that are no longer needed
del base_model
del peft_model
gc.collect()

# verify: the merged model should be a plain transformers model, not a PeftModel
print(f"model type after merge: {type(merged_model).__name__}")
print(f"total params: {sum(p.numel() for p in merged_model.parameters()):,}")

fusing LoRA weights into the base model...
model type after merge: CamembertForSequenceClassification
total params: 277,486,096


---
## 8 Saving the Merged Model

The merged model and tokenizer are saved to the output path using standard Hugging Face `save_pretrained` with `max_shard_size="2GB"` to split weights into smaller files. Sharding keeps peak memory during serialization well below the full model size and avoids OOM kills in memory constrained notebook environments. The resulting directory contains:

- `config.json`: model configuration including `id2label` and `label2id` mappings
- `model.safetensors` (or `pytorch_model.bin`): the merged weights
- `tokenizer.json`, `tokenizer_config.json` and vocab files: the tokenizer

This directory can be loaded with `AutoModelForSequenceClassification.from_pretrained(output_path)` without needing PEFT, or served directly with frameworks like vLLM, TGI or KServe.

In [9]:
print(f"saving merged model to {output_path}...")
merged_model.save_pretrained(output_path, max_shard_size="2GB")
tokenizer.save_pretrained(output_path)

# list the saved files
saved_files = os.listdir(output_path)
print(f"\nsaved {len(saved_files)} files:")
for fname in sorted(saved_files):
    fpath = os.path.join(output_path, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:40s} {size_mb:1f} MB")

print("\nmerge complete! the model is ready for standard Hugging Face inference.")

saving merged model to /mnt/data/adapter/merged_model...

saved 5 files:
  config.json                              0.002033 MB
  model.safetensors                        1058.548592 MB
  special_tokens_map.json                  0.000999 MB
  tokenizer.json                           16.545808 MB
  tokenizer_config.json                    0.137833 MB

merge complete! the model is ready for standard Hugging Face inference.


---
## 9 Quick Verification (Optional)

As a sanity check, reload the merged model from disk and run a forward pass to confirm it loads cleanly and produces sensible logits.

In [10]:
# free the merged model before reloading to avoid holding two copies
del merged_model
del tokenizer
gc.collect()

print("reloading merged model from disk for verification...")
reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    output_path, 
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(output_path, trust_remote_code=True)

# run a sample inference
test_text = "ขอเปิดใช้งานตอนนี้เลยค่ะ จะได้เริ่มใช้เบอร์ได้ แล้วช่วยดูโปรใหม่มีอะไรบ้างค่ะ ของเก่ารู้สึกโปรมันแพงไป T T"
inputs = reloaded_tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = reloaded_model(**inputs)

logits = outputs.logits
predicted_id = logits.argmax(dim=-1).item()
predicted_label = reloaded_model.config.id2label[predicted_id]

print(f"input:          {test_text}")
print(f"predicted id:   {predicted_id}")
print(f"predicted label: {predicted_label}")
print(f"logits shape:   {logits.shape} (batch_size x {num_labels} classes)")

reloading merged model from disk for verification...
input:          ขอเปิดใช้งานตอนนี้เลยค่ะ จะได้เริ่มใช้เบอร์ได้ แล้วช่วยดูโปรใหม่มีอะไรบ้างค่ะ ของเก่ารู้สึกโปรมันแพงไป T T
predicted id:   8
predicted label: MOBILE_SIM_ACTIVATION_POSTPAID
logits shape:   torch.Size([1, 16]) (batch_size x 16 classes)


---
## 10 Upload Merged Model to S3

Once the merged model is saved locally, we upload the entire output directory to an S3 bucket so it can be pulled by serving infrastructure or shared across environments.

Configuration is controlled by three environment variables:
- `S3_BUCKET`: the target S3 bucket name
- `S3_PREFIX`: the key prefix (folder path) inside the bucket
- `AWS_DEFAULT_REGION`: region for the S3 client (defaults to `us-east-1`)

Standard S3 credentials (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` or an instance profile) must be available in the environment.

In [11]:
import boto3
from botocore.config import Config

os.environ["AWS_ACCESS_KEY_ID"] = "minio" # your access key
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123" # your secret
os.environ["AWS_ENDPOINT_URL"] = "http://minio-service.minio.svc.cluster.local:9000" # url of the bucket --> http://s3.openshift-storage.svc.cluster.local

s3_bucket = os.environ.get("S3_BUCKET", "phayathaibert-model-bucket")
s3_prefix = os.environ.get("S3_PREFIX", "merged-models/encoder")
region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

s3 = boto3.client("s3", config=Config(region_name=region))

# walk the output directory and upload every file
uploaded = []
for root, _dirs, files in os.walk(output_path):
    for fname in sorted(files):
        local_path = os.path.join(root, fname)
        relative = os.path.relpath(local_path, output_path)
        s3_key = f"{s3_prefix}/{relative}"

        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"uploading {relative} ({size_mb:.1f} MB) -> s3://{s3_bucket}/{s3_key}")
        s3.upload_file(local_path, s3_bucket, s3_key)
        uploaded.append(s3_key)

print(f"\nuploaded {len(uploaded)} files to s3://{s3_bucket}/{s3_prefix}/")
print("done.")

uploading config.json (0.0 MB) -> s3://phayathaibert-model-bucket/merged-models/encoder/config.json
uploading model.safetensors (1058.5 MB) -> s3://phayathaibert-model-bucket/merged-models/encoder/model.safetensors
uploading special_tokens_map.json (0.0 MB) -> s3://phayathaibert-model-bucket/merged-models/encoder/special_tokens_map.json
uploading tokenizer.json (16.5 MB) -> s3://phayathaibert-model-bucket/merged-models/encoder/tokenizer.json
uploading tokenizer_config.json (0.1 MB) -> s3://phayathaibert-model-bucket/merged-models/encoder/tokenizer_config.json

uploaded 5 files to s3://phayathaibert-model-bucket/merged-models/encoder/
done.


---
## Summary

| Step | What it does |
|---|---|
| Label mapping | Scans training data to reconstruct the exact intent to id mapping used during fine tuning |
| Base model load | Loads the encoder with a classification head sized to match the adapter |
| Adapter load | Attaches the LoRA low rank matrices to the base model |
| Merge and unload | Fuses adapter deltas into the base weights via `W + (alpha/r) * B @ A` |
| Save | Writes a standalone model directory that works without PEFT |
| Verify | Reloads from disk and runs a forward pass to confirm everything is correct |
| Upload to S3 | Pushes the full merged checkpoint to an S3 bucket via boto3 |

The merged model is now ready for deployment.